# Get the image urls from metadata

Make sure each url is valid(reuse function from caption classification notebook)

Rewrite the query model function slightly to support specifying prompt location.

In [1]:
from openrouter_client import *
from roboflow_client import *

import dotenv
import os
import json
import time
import tqdm
import base64
from io import BytesIO
from PIL import Image
import requests
from tqdm.notebook import tqdm as tqdmnote
import openai as opi

import base64
import re
from typing import Optional, Tuple

import asyncio


from threading import Thread, Event

for key in os.environ.keys():
    os.environ.pop(key)

dotenv.load_dotenv()

True

In [2]:
with open("./our-dataset-captions.json", "r", encoding="utf-8") as file:
    our_metadata = json.load(file)

# load VERI
with open("./VERI-dataset-captions.json", "r", encoding="utf-8") as file:
    veri_metadata = json.load(file)
for i, img in enumerate(veri_metadata):
    veri_metadata[img]["label"] = "00" if veri_metadata[img]["label"] == "0" else "11"

In [7]:
# Converts invalid image formats to correct formats
def get_openai_image_url(url):
    try:
        # 1. Download the data to check the actual format
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        
        # 2. Inspect the data (not the extension)
        img = Image.open(BytesIO(resp.content))
        actual_format = img.format.upper() if img.format else ""

        # 3. If it's already a standard web format, just return the URL
        if actual_format in ['PNG', 'JPEG', 'WEBP']:
            return url

        # 4. Otherwise (TIFF, BMP, etc.), convert to PNG and encode to Base64
        buffer = BytesIO()
        if img.mode not in ("RGB", "RGBA"):
            img = img.convert("RGBA")
        
        img.save(buffer, format="PNG")
        b64_data = base64.b64encode(buffer.getvalue()).decode('utf-8')
        
        return f"data:image/png;base64,{b64_data}"

    except Exception as e:
        return f"Error processing image: {e}"


def classify_image(model, template_prompt, *urls):
    """
    Query OpenRouter API for image classification.
    
    Returns: (predicted_code, full_response)
    """
    try:
        # Make vision API call with text prompt and image
        client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY")
        )
        prompts = template_prompt.split("__IMAGE__")
        objects = []
        for i, url in enumerate(urls):
            objects.append({"type": "text", "text": prompts[i]})
            objects.append({"type": "image_url", "image_url": get_openai_image_url(url)})
        objects.append({"type": "text", "text": prompts[-1]})
        
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": objects
                }
            ],
            max_tokens=200  # Limit response length since we only need 2 digits
        )
        
        # Extract model's text response
        full_response = response.choices[0].message.content.strip()
        
        # Parse 2-digit code from response text
        predicted_code = extract_code(full_response)
        
        return predicted_code, full_response
    except Exception as e:
        print(f"Error querying OpenRouter API: {e}")
        return None, str(e)

def extract_code(response):
    """Extract 2-digit code from model response."""
    # Use regex to find first 2-digit number (word boundaries ensure exact match)
    match = re.search(r'\b\d{2}\b', response[-100:])
    if match:
        # Return the matched 2-digit code
        return match.group(0)
    return None

# Performs multi-query run
def run_prompt_sequence(model, prompt_set, img_data, outputs, prompt_urls = None):
    if len(prompt_set) == 0:
        print("No prompt!")
        return ""
    # unpack the img data

    model_prediction = ""
    model_response = ""
    caption = img_data["caption"]
    img_id = img_data["id"]
    img_url = img_data["url"]
    img_label = img_data["label"]
    urls = prompt_urls[:] if prompt_urls else []
    
    urls.append(img_url)

    history = ""
    # for each prompt in the set, takes any past response, appends it, and asks with next prompt
    # Kind of self-generating few shot prompt
    for prompt in prompt_set:
        history += "\n" + prompt + "\n"
        history += "\n Response: "
        # print(f"Using history for model {model}: \n {history}")
        model_prediction, model_response = classify_image(model, history, *urls)
        # print("==="*30)
        history += model_response

    outputs.append({
        "image_id": img_id,
        "image_url": img_url,
        "ground_truth": img_label,
        "model_prediction": model_prediction,
        "model_response": model_response,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "match": model_prediction == img_label
    })

stop_model_thread = Event()
def run_model(model, data, prompt_set, class_limit = None, example_urls = None):
    print(f"Running model {model}")
    run_outputs = []
    limits = {}
    try:
        bar = tqdmnote(data)
        for i, image in enumerate(bar):
            if(stop_model_thread.is_set()): break

            sample = data[image]
            label = sample["label"]
            if class_limit and label in limits and limits[label] >= class_limit:
                print("Reached limit for class", label)
                continue
            limits[label] = 1 if label not in limits else limits[label] + 1 
            sample["id"] = image
            run_prompt_sequence(model, prompt_set, sample, run_outputs, example_urls)
    except Exception as e:
        print(e)
        pass # just move on and output current results
    
    output = {"prompts": prompt_set, "results": run_outputs}
    save_output(model, output)

def save_output(model, output):
    # Save the file
    output_dir = f"./results/{model.replace("/", "_")}_run"
    i = 0;
    while(os.path.isdir(output_dir + f"{i:03d}")):
        i += 1
    output_dir +=  f"{i:03d}"
    os.mkdir(output_dir)
    with open(f"{output_dir}/results.json", "w", encoding="utf-8") as file:
        json.dump(output, file, indent=4)

    print(f"Results for {model} saved to {output_dir}/results.json")

In [8]:
cot_prompt = [
    """You are inspecting an environment for anomalies and hazards to help first responders. Use the following definitions. Hazard: A scene is hazardous if normal interaction with at least one of the present elements would realistically cause immediate physical harm to the people or put the existing property in significant danger. Anomaly: A scene is anomalous if any visible element clearly violates normal expectations for this environment (location, arrangement, state, or use). Safe: Safe scenes include normal, expected human activities in the specifically given appropriate environment, provided no immediate physical hazard is present. Be concise and respond to the following:

    In accordance with the definition for "Hazard," identify any real physically present element that could cause immediate harm. Also consider if people or property are directly in danger.
    __IMAGE__
    """,
    """
    Independently determine whether any element violates normal expectations for this environment using the definition for "Anomaly".
    """,
    """
    Decide whether anomaly and hazard are present independently. Make a final decision and output a single code. You MUST obey this strict output format: `<a brief, explicit justification for the final decision>: <code>` Where <code> is ONLY one of: 00 = Safe, 10 = Anomalous, 01 = Hazardous 11 = Anomalous AND Hazardous - No extra paragraphs, no bullet points, no additional commentary.
    """
]

fs_prompt = [
"""
You are inspecting an environment for anomalies and hazards to help first responders. Use the following definitions. Hazard: A scene is hazardous if normal interaction with at least one of the present elements would realistically cause immediate physical harm to the people or put the existing property in significant danger. Anomaly: A scene is anomalous if any visible element clearly violates normal expectations for this environment (location, arrangement, state, or use). Safe: Safe scenes include normal, expected human activities in the specifically given appropriate environment, provided no immediate physical hazard is present.
Decide whether anomaly and hazard are present independently. Make a final decision and output a single code. You MUST obey this strict output format: `<a brief, explicit justification for the final decision>: <code>` Where <code> is ONLY one of: 00 = Safe, 10 = Anomalous, 01 = Hazardous 11 = Anomalous AND Hazardous - No extra paragraphs, no bullet points, no additional commentary.
Image:
__IMAGE__
Response:
First, considering hazard: the kitchen seems to be a safe environment with no described elements posing significant danger. Next considering anomaly: the kitchen seems to be clean and organized. It is in a normal state. Code : 00

You are inspecting an environment for anomalies and hazards to help first responders. Use the following definitions. Hazard: A scene is hazardous if normal interaction with at least one of the present elements would realistically cause immediate physical harm to the people or put the existing property in significant danger. Anomaly: A scene is anomalous if any visible element clearly violates normal expectations for this environment (location, arrangement, state, or use). Safe: Safe scenes include normal, expected human activities in the specifically given appropriate environment, provided no immediate physical hazard is present.
Decide whether anomaly and hazard are present independently. Make a final decision and output a single code. You MUST obey this strict output format: `<a brief, explicit justification for the final decision>: <code>` Where <code> is ONLY one of: 00 = Safe, 10 = Anomalous, 01 = Hazardous 11 = Anomalous AND Hazardous - No extra paragraphs, no bullet points, no additional commentary.
Image:
__IMAGE__
Response:
First, considering hazard: the men are working on high voltage power lines. There is significant risk of getting electrocuted and being seriously injured. Next considering anomaly: the workers seem to be doing their normal job. Code : 01

You are inspecting an environment for anomalies and hazards to help first responders. Use the following definitions. Hazard: A scene is hazardous if normal interaction with at least one of the present elements would realistically cause immediate physical harm to the people or put the existing property in significant danger. Anomaly: A scene is anomalous if any visible element clearly violates normal expectations for this environment (location, arrangement, state, or use). Safe: Safe scenes include normal, expected human activities in the specifically given appropriate environment, provided no immediate physical hazard is present.
Decide whether anomaly and hazard are present independently. Make a final decision and output a single code. You MUST obey this strict output format: `<a brief, explicit justification for the final decision>: <code>` Where <code> is ONLY one of: 00 = Safe, 10 = Anomalous, 01 = Hazardous 11 = Anomalous AND Hazardous - No extra paragraphs, no bullet points, no additional commentary.
Image:
__IMAGE__
Response:
First considering hazard: there are no indications of immediate and significant danger. Next considering anomaly: the man seems to have an atypical expression on his face suggesting something is anomalous. Code : 10

You are inspecting an environment for anomalies and hazards to help first responders. Use the following definitions. Hazard: A scene is hazardous if normal interaction with at least one of the present elements would realistically cause immediate physical harm to the people or put the existing property in significant danger. Anomaly: A scene is anomalous if any visible element clearly violates normal expectations for this environment (location, arrangement, state, or use). Safe: Safe scenes include normal, expected human activities in the specifically given appropriate environment, provided no immediate physical hazard is present.
Decide whether anomaly and hazard are present independently. Make a final decision and output a single code. You MUST obey this strict output format: `<a brief, explicit justification for the final decision>: <code>` Where <code> is ONLY one of: 00 = Safe, 10 = Anomalous, 01 = Hazardous 11 = Anomalous AND Hazardous - No extra paragraphs, no bullet points, no additional commentary.
Image:
__IMAGE__
Response:
First considering hazard: the fire is widespread and not controlled, posing significant and imminent danger to people. Next considering anomaly: forests should not normally be on fire. Code: 11

You are inspecting an environment for anomalies and hazards to help first responders. Use the following definitions. Hazard: A scene is hazardous if normal interaction with at least one of the present elements would realistically cause immediate physical harm to the people or put the existing property in significant danger. Anomaly: A scene is anomalous if any visible element clearly violates normal expectations for this environment (location, arrangement, state, or use). Safe: Safe scenes include normal, expected human activities in the specifically given appropriate environment, provided no immediate physical hazard is present.
Decide whether anomaly and hazard are present independently. Make a final decision and output a single code. You MUST obey this strict output format: `<a brief, explicit justification for the final decision>: <code>` Where <code> is ONLY one of: 00 = Safe, 10 = Anomalous, 01 = Hazardous 11 = Anomalous AND Hazardous - No extra paragraphs, no bullet points, no additional commentary.
Image:
__IMAGE__
"""
]

zero_shot_prompt = [
"""
You are inspecting an environment for anomalies and hazards to help first responders. Use the following definitions. Hazard: A scene is hazardous if normal interaction with at least one of the present elements would realistically cause immediate physical harm to the people or put the existing property in significant danger. Anomaly: A scene is anomalous if any visible element clearly violates normal expectations for this environment (location, arrangement, state, or use). Safe: Safe scenes include normal, expected human activities in the specifically given appropriate environment, provided no immediate physical hazard is present.
Identify hazards or anomalies using the given definitions. Use that conclusion to identify points of interest needing immediate attention and justify the classification concisely.
(1) Identify concrete sources of immediate harm.
(3) Independently assess contextual irregularities.
(4) Decide hazard and anomaly labels independently.
(5) Obey this STRICT output format: `<a brief, explicit justification for the final decision>: <code>` Where <code> is ONLY one of: 00 = Safe, 10 = Anomalous, 01 = Hazardous 11 = Anomalous AND Hazardous - No extra paragraphs, no bullet points, no additional commentary.
"""
]

bsts_prompt = ["""
Does the situation in this image appear dangerous or safe?
(5) Obey this STRICT output format: `<a brief, explicit justification for the final decision>: <code>` Where <code> is ONLY one of: 00 = Safe, Hazardous 11 = Hazardous - No extra paragraphs, no bullet points, no additional commentary.
"""]

models = ["mistralai/ministral-3b-2512", "mistralai/ministral-14b-2512", "mistralai/mistral-large-2512", "google/gemini-3-flash-preview", "google/gemini-2.5-flash", "openai/gpt-4.1"]

# COT Run

In [9]:
threads = [Thread(target=run_model, args=(model, veri_metadata, cot_prompt)) for model in models]

stop_model_thread.clear()
for thread in threads:
    thread.start()
try:
    pass
except:
    print("Exception occurred, waiting threads to finish...")
    stop_model_thread.set()
finally:
    for thread in threads:
        thread.join()

Running model mistralai/ministral-3b-2512
Running model mistralai/ministral-14b-2512
Running model mistralai/mistral-large-2512


  0%|          | 0/200 [00:00<?, ?it/s]

Running model google/gemini-3-flash-preview
Running model google/gemini-2.5-flash
Running model openai/gpt-4.1


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

Error querying OpenRouter API: 'NoneType' object has no attribute 'strip'
Results for mistralai/ministral-3b-2512 saved to ./results/mistralai_ministral-3b-2512_run001/results.json
Results for google/gemini-2.5-flash saved to ./results/google_gemini-2.5-flash_run001/results.json
Error querying OpenRouter API: 'NoneType' object has no attribute 'strip'
Error querying OpenRouter API: 'NoneType' object has no attribute 'strip'
Results for openai/gpt-4.1 saved to ./results/openai_gpt-4.1_run001/results.json
Results for mistralai/ministral-14b-2512 saved to ./results/mistralai_ministral-14b-2512_run001/results.json
Results for google/gemini-3-flash-preview saved to ./results/google_gemini-3-flash-preview_run001/results.json
Results for mistralai/mistral-large-2512 saved to ./results/mistralai_mistral-large-2512_run001/results.json


# Few Shot Run

In [10]:

threads = [Thread(target=run_model, args=(model, veri_metadata, fs_prompt), kwargs={
    "example_urls": [
        our_metadata["HrWjsGe2Eqn2W9BfkG7v"]["url"],
        our_metadata["EFxNryFxTwhgbaIBDkt5"]["url"],
        our_metadata["t5x2veqCS7NkqUCIPejF"]["url"],
        our_metadata["CHl2oSfh74sEwqLFwKcy"]["url"]
    ]}) for model in models]

stop_model_thread.clear()
for thread in threads:
    thread.start()
try:
    pass
except:
    print("Exception occurred, waiting threads to finish...")
    stop_model_thread.set()
finally:
    for thread in threads:
        thread.join()

Running model mistralai/ministral-3b-2512Running model mistralai/ministral-14b-2512Running model mistralai/mistral-large-2512
Running model google/gemini-3-flash-preview

Running model google/gemini-2.5-flash

Running model openai/gpt-4.1


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

Results for google/gemini-2.5-flash saved to ./results/google_gemini-2.5-flash_run002/results.json
Results for mistralai/ministral-14b-2512 saved to ./results/mistralai_ministral-14b-2512_run002/results.json
Results for openai/gpt-4.1 saved to ./results/openai_gpt-4.1_run002/results.json
Results for google/gemini-3-flash-preview saved to ./results/google_gemini-3-flash-preview_run002/results.json
Results for mistralai/mistral-large-2512 saved to ./results/mistralai_mistral-large-2512_run002/results.json
Results for mistralai/ministral-3b-2512 saved to ./results/mistralai_ministral-3b-2512_run002/results.json


# Zero Shot Run

In [11]:

threads = [Thread(target=run_model, args=(model, veri_metadata, zero_shot_prompt)) for model in models]

stop_model_thread.clear()
for thread in threads:
    thread.start()
try:
    pass
except:
    print("Exception occurred, waiting threads to finish...")
    stop_model_thread.set()
finally:
    for thread in threads:
        thread.join()

Running model mistralai/ministral-3b-2512Running model mistralai/ministral-14b-2512Running model mistralai/mistral-large-2512Running model google/gemini-3-flash-preview



Running model google/gemini-2.5-flash
Running model openai/gpt-4.1


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

Results for mistralai/ministral-3b-2512 saved to ./results/mistralai_ministral-3b-2512_run003/results.json
Results for openai/gpt-4.1 saved to ./results/openai_gpt-4.1_run003/results.json
Results for google/gemini-2.5-flash saved to ./results/google_gemini-2.5-flash_run003/results.json
Results for mistralai/mistral-large-2512 saved to ./results/mistralai_mistral-large-2512_run003/results.json
Results for google/gemini-3-flash-preview saved to ./results/google_gemini-3-flash-preview_run003/results.json
Results for mistralai/ministral-14b-2512 saved to ./results/mistralai_ministral-14b-2512_run003/results.json


# COT Image Run Compare with VERI Dataset

In [12]:
threads = [Thread(target=run_model, args=(model, veri_metadata, bsts_prompt)) for model in models]

stop_model_thread.clear()
for thread in threads:
    thread.start()
try:
    pass
except:
    print("Exception occurred, waiting threads to finish...")
    stop_model_thread.set()
finally:
    for thread in threads:
        thread.join()

Running model mistralai/ministral-3b-2512Running model mistralai/ministral-14b-2512Running model google/gemini-3-flash-previewRunning model mistralai/mistral-large-2512Running model google/gemini-2.5-flash




Running model openai/gpt-4.1


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

Results for mistralai/ministral-3b-2512 saved to ./results/mistralai_ministral-3b-2512_run004/results.json
Results for openai/gpt-4.1 saved to ./results/openai_gpt-4.1_run004/results.json
Results for mistralai/mistral-large-2512 saved to ./results/mistralai_mistral-large-2512_run004/results.json
Results for google/gemini-2.5-flash saved to ./results/google_gemini-2.5-flash_run004/results.json
Results for google/gemini-3-flash-preview saved to ./results/google_gemini-3-flash-preview_run004/results.json
Results for mistralai/ministral-14b-2512 saved to ./results/mistralai_ministral-14b-2512_run004/results.json
